In [ ]:

!pip install -q x-transformers==1.16.23 einops==0.6.1 gin-config pytorch-ignite==0.5.0.post2 pytorch-lightning torchmetrics
!git clone -q -b development https://github.com/Katja-Jagd/YAIB.git /content/YAIB


In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import sys, os
sys.path.insert(0, "/content/YAIB")
D = "/content/drive/MyDrive/BAT_colab/"
TAG = "_ivEicu"
VTAG = "_logtarget_ivEicu"   # variant outputs

needed = ["bat_X_scaled.npy", "bat_static_scaled.npy", "bat_mask.npy", "bat_y.npy",
          "bat_train_rows.npy", "bat_val_rows.npy", "bat_test_rows.npy"]
missing = [f for f in needed if not os.path.exists(D + f)]
print("Missing files:", missing if missing else "none, all data found")

from importlib.metadata import version
print("x-transformers version:", version("x-transformers"), "(must be 1.16.23)")

Mounted at /content/drive
Missing files: none, all data found
x-transformers version: 1.16.23 (must be 1.16.23)


In [ ]:

import urllib.request, torch
from icu_benchmarks.models.dl_models.bat import BAT, RegressionHead

CKPT = D + "bat_pretrained_eicu_mimiciv.ckpt"
if not os.path.exists(CKPT):
    urllib.request.urlretrieve(
        "https://huggingface.co/Katja-Jagd/bat-pretrained-eicu-mimiciv-base/resolve/main/model.ckpt",
        CKPT)

ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)
hp = ckpt["hyper_parameters"]
encoder_weights = {k: v for k, v in ckpt["state_dict"].items()
                   if k.startswith("model.encoder_class.")}

def build_model():
    m = BAT(
        input_size=(64, 48, 24),
        value_embed_size=hp["value_embed_size"],   # 64
        layers=hp["layers"],                       # 6
        heads=hp.get("heads", 1),
        dropout=hp["dropout"],                     # 0.34844
        attn_dropout=hp["attn_dropout"],           # 0.28413
        use_mask=False,
        prediction_head=RegressionHead,
        prediction_head_kwargs={"output_dim": 1},
        static_count=4,
    )
    r = m.load_state_dict(encoder_weights, strict=False)
    return m, r

model, result = build_model()
print("Settings:", {k: hp[k] for k in ["value_embed_size", "layers", "dropout", "attn_dropout"]})
print("Missing keys (expect 4, the new head):", result.missing_keys)
print("Unexpected keys (must be empty):", result.unexpected_keys)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}  (laptop check said 846,469)")
assert len(result.unexpected_keys) == 0, "STOP: checkpoint did not load cleanly"



use static
Settings: {'value_embed_size': 64, 'layers': 6, 'dropout': 0.34844, 'attn_dropout': 0.28413}
Missing keys (expect 4, the new head): ['model.head.linear.weight', 'model.head.linear.bias', 'logit.weight', 'logit.bias']
Unexpected keys (must be empty): []
Trainable parameters: 846,469  (laptop check said 846,469)


In [ ]:

import numpy as np
from torch.utils.data import TensorDataset, DataLoader

X = np.load(D + "bat_X_scaled.npy")          # (51835, 24, 48)
static = np.load(D + "bat_static_scaled.npy")
mask = np.load(D + "bat_mask.npy")
y = np.load(D + "bat_y.npy")
train_rows = np.load(D + "bat_train_rows.npy")
val_rows = np.load(D + "bat_val_rows.npy")
test_rows = np.load(D + "bat_test_rows.npy")
print("Split sizes:", len(train_rows), len(val_rows), len(test_rows), "(expect 31101, 10367, 10367)")

def build_tensors(idx):
    return (torch.tensor(np.transpose(X[idx], (0, 2, 1)), dtype=torch.float32),   # (n, 48, 24)
            torch.tensor(static[idx], dtype=torch.float32),
            torch.tensor(np.tile(np.arange(24, dtype=np.float32), (len(idx), 1))),
            torch.tensor(np.transpose(mask[idx], (0, 2, 1)), dtype=torch.float32),
            torch.tensor(np.log1p(y[idx]), dtype=torch.float32))   # the only change: log(1 + LOS)

torch.manual_seed(42); np.random.seed(42)
train_loader = DataLoader(TensorDataset(*build_tensors(train_rows)), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(*build_tensors(val_rows)), batch_size=64)
test_loader = DataLoader(TensorDataset(*build_tensors(test_rows)), batch_size=64)

Split sizes: 31101 10367 10367 (expect 31101, 10367, 10367)


In [ ]:

import time

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.MSELoss()

BEST = D + f"bat_best_model{VTAG}.pt"
RESUME = D + f"bat_resume{VTAG}.pt"
LOG = D + f"bat_training_log{VTAG}.txt"
MAX_EPOCHS, PATIENCE = 200, 10

start_epoch, best_val, bad_epochs = 0, float("inf"), 0
if os.path.exists(RESUME):
    s = torch.load(RESUME, map_location=device, weights_only=False)
    model.load_state_dict(s["model"]); optimizer.load_state_dict(s["optimizer"])
    start_epoch, best_val, bad_epochs = s["epoch"], s["best_val"], s["bad_epochs"]
    print(f"Resuming after epoch {start_epoch} (best val so far {best_val:.4f})")

def log(msg):
    print(msg)
    with open(LOG, "a") as f:
        f.write(msg + "\n")

if start_epoch == 0:
    log(f"Starting BAT LOG-TARGET training (MIMIC-IV + eICU checkpoint) on {device} - max {MAX_EPOCHS} epochs, patience {PATIENCE}")

t0 = time.time()
for epoch in range(start_epoch, MAX_EPOCHS):
    model.train(); tr = 0
    for xb, sb, tb, mb, yb in train_loader:
        xb, sb, tb, mb, yb = [t.to(device) for t in (xb, sb, tb, mb, yb)]
        optimizer.zero_grad()
        loss = loss_fn(model(xb, sb, tb, mb).squeeze(-1), yb)
        loss.backward(); optimizer.step(); tr += loss.item()

    model.eval(); va = 0
    with torch.no_grad():
        for xb, sb, tb, mb, yb in val_loader:
            xb, sb, tb, mb, yb = [t.to(device) for t in (xb, sb, tb, mb, yb)]
            va += loss_fn(model(xb, sb, tb, mb).squeeze(-1), yb).item()
    tr, va = tr / len(train_loader), va / len(val_loader)
    log(f"Epoch {epoch+1} - train loss: {tr:.4f} | val loss: {va:.4f} | elapsed: {(time.time()-t0)/60:.1f} min")

    if va < best_val:
        best_val, bad_epochs = va, 0
        torch.save(model.state_dict(), BEST)
        log(f"  -> New best val loss ({va:.4f}), model saved.")
    else:
        bad_epochs += 1

    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "epoch": epoch + 1, "best_val": best_val, "bad_epochs": bad_epochs}, RESUME)

    if bad_epochs >= PATIENCE:
        log(f"\nEarly stopping triggered after {epoch+1} epochs (no improvement for {PATIENCE} epochs).")
        break

log(f"\nTraining finished. Best val loss (log space): {best_val:.4f}")

Starting BAT LOG-TARGET training (MIMIC-IV + eICU checkpoint) on cuda - max 200 epochs, patience 10
Epoch 1 - train loss: 0.3592 | val loss: 0.3430 | elapsed: 3.6 min
  -> New best val loss (0.3430), model saved.
Epoch 2 - train loss: 0.2857 | val loss: 0.3414 | elapsed: 7.1 min
  -> New best val loss (0.3414), model saved.
Epoch 3 - train loss: 0.2769 | val loss: 0.3130 | elapsed: 10.7 min
  -> New best val loss (0.3130), model saved.
Epoch 4 - train loss: 0.2720 | val loss: 0.3279 | elapsed: 14.3 min
Epoch 5 - train loss: 0.2689 | val loss: 0.3145 | elapsed: 17.9 min
Epoch 6 - train loss: 0.2659 | val loss: 0.3203 | elapsed: 21.4 min
Epoch 7 - train loss: 0.2647 | val loss: 0.3200 | elapsed: 25.0 min
Epoch 8 - train loss: 0.2603 | val loss: 0.2989 | elapsed: 28.6 min
  -> New best val loss (0.2989), model saved.
Epoch 9 - train loss: 0.2603 | val loss: 0.3023 | elapsed: 32.2 min
Epoch 10 - train loss: 0.2572 | val loss: 0.2725 | elapsed: 35.7 min
  -> New best val loss (0.2725), mode

In [ ]:
model.load_state_dict(torch.load(BEST, map_location=device, weights_only=False))
model.eval(); preds_log, targs_log = [], []
with torch.no_grad():
    for xb, sb, tb, mb, yb in test_loader:
        xb, sb, tb, mb = [t.to(device) for t in (xb, sb, tb, mb)]
        preds_log.append(model(xb, sb, tb, mb).squeeze(-1).cpu().numpy())
        targs_log.append(yb.numpy())
preds = np.expm1(np.concatenate(preds_log).astype(np.float64))     # back to days
targs = np.expm1(np.concatenate(targs_log).astype(np.float64))
np.save(D + f"bat{VTAG}_test_predictions_days.npy", preds.astype(np.float32))
np.save(D + f"bat{VTAG}_test_targets_days.npy", targs.astype(np.float32))

err = preds - targs
log(f"\n=== FINAL TEST SET RESULTS (BAT log-target, MIMIC-IV + eICU checkpoint, in days) ===")
log(f"MAE: {np.abs(err).mean():.4f} days")
log(f"RMSE: {np.sqrt((err**2).mean()):.4f} days")
log(f"Mean error: {err.mean():.4f} days")
log(f"Dangerous cases (error <= -3 days): {(err <= -3).sum()} ({(err <= -3).mean()*100:.2f}%)")

# Why: the targets went through log and back, so check they still equal the
# original lengths of stay, in the same patient order as every other file.
orig_t = np.load(D + "bat_test_targets.npy").ravel().astype(np.float64)
print("Test targets match original run (after converting back):", np.allclose(orig_t, targs, atol=1e-3))
print("For comparison, main new BAT: MAE 2.4708, RMSE 5.4190, mean error -0.4165, D(-3) 11.47%")


=== FINAL TEST SET RESULTS (BAT log-target, MIMIC-IV + eICU checkpoint, in days) ===
MAE: 2.2604 days
RMSE: 5.6152 days
Mean error: -1.3187 days
Dangerous cases (error <= -3 days): 1449 (13.98%)
Test targets match original run (after converting back): True
For comparison, main new BAT: MAE 2.4708, RMSE 5.4190, mean error -0.4165, D(-3) 11.47%
